In [3]:
from scipy.io import loadmat
import numpy as np
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [4]:
data = loadmat("sub1_comp.mat")
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'train_data', 'test_data', 'train_dg'])


In [5]:
data["train_data"].shape

(400000, 62)

In [6]:
data["train_dg"].shape

(400000, 5)

**Windowing Step:**

In the windowing step, we take a fixed number of time steps from the continuous data and convert them into meaningful samples that can be used to train the model.

For example, if we choose a window size of 100 time steps, then every 100 consecutive time steps are grouped together to form one sample. This sample is then used as input to the training model.

This helps the model learn patterns over time instead of relying on a single time step.


In [7]:
# Windowing Step
def sequences(X, y, window_size= 100):
  X_seq = []
  y_seq = []

  for i in range(window_size,len(X)):
    X_seq.append(X[i - window_size:i])
    y_seq.append(y[i])

  return np.array(X_seq), np.array(y_seq)

X = data["train_data"]
y = data["train_dg"]

X_seq, y_seq = sequences(X, y, window_size=100)

print(X_seq.shape)
print(y_seq.shape)

(399900, 100, 62)
(399900, 5)


In [8]:
# Feature Extraction
def feature_extraction(X_seq):
  feature_list = []

  for seq in X_seq:
    mean_feature = np.mean(seq, axis = 0)
    std_feature = np.std(seq, axis = 0)
    min_feature = np.min(seq, axis = 0)
    max_feature = np.max(seq, axis = 0)

    features = np.concatenate([mean_feature, std_feature, min_feature, max_feature])
    feature_list.append(features)

  return np.array(feature_list)

X_feature = feature_extraction(X_seq)

print(X_feature.shape)
print(y_seq.shape)

(399900, 248)
(399900, 5)


In [9]:
# Scaling the Feature
x_batch = X_feature[:50000]
y_batch = y_seq[:50000]

Scaler = StandardScaler()
X_scaled = Scaler.fit_transform(x_batch)

print(X_scaled.shape)
print(y_batch.shape)




(50000, 248)
(50000, 5)


In [10]:
channel_names = [f"ch_{i}" for i in range(62)]

feature_names = []

for ch in channel_names:
    feature_names.append(f"{ch}_mean")
    feature_names.append(f"{ch}_std")
    feature_names.append(f"{ch}_min")
    feature_names.append(f"{ch}_max")

print(len(feature_names))
print(feature_names[:10])

248
['ch_0_mean', 'ch_0_std', 'ch_0_min', 'ch_0_max', 'ch_1_mean', 'ch_1_std', 'ch_1_min', 'ch_1_max', 'ch_2_mean', 'ch_2_std']


In [11]:
# convert np arrays into PD dataframes
data_df = pd.DataFrame(X_scaled, columns = feature_names)

print(data_df.head())

   ch_0_mean  ch_0_std  ch_0_min  ch_0_max  ch_1_mean  ch_1_std  ch_1_min  \
0  -0.797038  1.018936 -1.482072  0.673665  -0.167744  0.997015  1.281356   
1  -0.789382  1.025136 -1.459524  0.648379  -0.175290  0.981979  1.281693   
2  -0.779134  1.030945 -1.435986  0.622884  -0.184149  0.966043  1.279267   
3  -0.768002  1.036241 -1.413370  0.597282  -0.194474  0.950575  1.274897   
4  -0.756908  1.040434 -1.394046  0.571747  -0.205774  0.936612  1.268635   

   ch_1_max  ch_2_mean  ch_2_std  ...  ch_59_min  ch_59_max  ch_60_mean  \
0  0.303869  -0.109389 -0.295555  ...   1.441615  -0.119035    1.738909   
1  0.305804  -0.121371 -0.312935  ...   1.441615  -0.119035    1.738909   
2  0.307451  -0.133983 -0.330107  ...   1.441615  -0.119035    1.738909   
3  0.308819  -0.146970 -0.346209  ...   1.441615  -0.119035    1.738909   
4  0.309969  -0.159688 -0.360881  ...   1.441615  -0.119035    1.738909   

   ch_60_std  ch_60_min  ch_60_max  ch_61_mean  ch_61_std  ch_61_min  \
0   0.192668  